In [2]:
%load_ext autoreload
%autoreload 2


In [3]:
%reload_ext autoreload


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from skfda import FDataGrid
from skfda.representation.basis import BSplineBasis
from skfda.preprocessing.smoothing import BasisSmoother
from skfda.misc.regularization import L2Regularization
from skfda.misc.operators import LinearDifferentialOperator


from src.Smooth_Method.Smoothing_Spline import registration, lambda_finding, compute_smoothed_train
from src.MFPCA import run_mfpca
from src.Label import fit_gmm, label_merged, filter_candidates, take_iv, Youden_merged
from src.Smooth_Method.Regression_Spline import compute_smoothed_test
from src.Youden import knn_apply
from src.result_check import check_rmse, sweep_k
sensor_names = [
    "T2", "T24", "T30", "T50", "P2", "P15", "P30", "Nf", "Nc", "epr",
    "Ps30", "phi", "NRf", "NRc", "BPR", "farB", "htBleed", "Nfdmd", "PCNfRdmd", "W31", "W32"
]
base_names = ['unit_number', 'cycles', 'op1', 'op2', 'op3']
sensor_columns = base_names + sensor_names 

base_keep_names = ['unit_number', 'cycles']
SENSORS = ["T24", "T30", "T50", "P30", "Ps30", "phi", "BPR", "W31", "W32"]
sensor_keep_columns = base_keep_names + SENSORS
train_df = pd.read_csv('Data/train_FD001.txt', sep='\\s+', header=None, names=sensor_columns)
train_df = train_df[sensor_keep_columns]

test_df = pd.read_csv('Data/test_FD001.txt', sep='\\s+', header=None, names=sensor_columns)
test_df = test_df[sensor_keep_columns]

check_df = pd.read_csv('Data/RUL_FD001.txt', sep='\\s+', header=None, names=['RUL'])
check_df['unit_number'] = np.arange(1, len(check_df) + 1)



In [4]:
train_df = registration(train_df)

In [6]:
summary_lambdas, unit_cache = lambda_finding(train_df, SENSORS, cycle_col="t_registered")

  sensor    lambda        gcv
0    T24  0.012581   0.093138
1    T30  0.014806  16.366782
2    T50  0.005430  16.587771
3    P30  0.005976   0.170488
4   Ps30  0.004022   0.010600
5    phi  0.004822   0.093728
6    BPR  0.008878   0.000413
7    W31  0.010000   0.010471
8    W32  0.010113   0.003694


In [7]:
train_smoothed = compute_smoothed_train(train_df, summary_lambdas, unit_cache, SENSORS)

Smoothed Dataframe :
       unit_number  t_registered         T24          T30          T50  \
0                1      0.000000  642.289657  1587.434701  1400.757907   
1                1      0.005236  642.291533  1587.410602  1400.722768   
2                1      0.010471  642.293403  1587.386568  1400.687608   
3                1      0.015707  642.295262  1587.362617  1400.652604   
4                1      0.020942  642.297102  1587.338764  1400.617936   
...            ...           ...         ...          ...          ...   
20626          100      0.979899  643.483922  1600.628724  1427.493349   
20627          100      0.984925  643.496300  1600.781270  1427.825041   
20628          100      0.989950  643.508681  1600.933899  1428.157030   
20629          100      0.994975  643.521066  1601.086555  1428.489083   
20630          100      1.000000  643.533456  1601.239188  1428.820966   

              P30       Ps30         phi       BPR        W31        W32  
0      554.1326

In [8]:
rho_score, exp_var = run_mfpca(train_smoothed, SENSORS)

   mfpc  explained_var_pct  cumulative_pct
0     1          95.084146       95.084146
1     2           2.107639       97.191786
2     3           0.753925       97.945710
3     4           0.481677       98.427387
4     5           0.427724       98.855111
5     6           0.344398       99.199509
6     7           0.303259       99.502769
7     8           0.193182       99.695951
8     9           0.160539       99.856490
9    10           0.072833       99.929323


In [9]:
label_train = fit_gmm(train_smoothed, rho_score)

label
0    65
1    35
Name: count, dtype: int64


In [10]:
iv_train = take_iv(train_smoothed, SENSORS)

In [11]:
youden_info = Youden_merged(label_train, iv_train)

In [12]:
iv_test = take_iv(test_df, SENSORS)

In [13]:
label_test = label_merged(test_df, iv_test, youden_info, SENSORS)

label
0    62
1    38
Name: count, dtype: int64


In [14]:
candidates = filter_candidates(train_df, test_df, label_train, label_test)

[filter_candidates] 1/100 units fell back to the opposite group due to insufficient candidates :
  - unit 49: original_group=1, n_own_group=35, n_opposite_valid=4


In [15]:
smoothed_test = compute_smoothed_test(candidates, train_df, test_df, SENSORS)

In [17]:
result = knn_apply(smoothed_test, train_df, candidates, k=8)

In [18]:
compare_df, rmse = check_rmse(result, check_df)

In [19]:
# compare_df[compare_df['unit_number']==20]
compare_df.head(20)

,unit_number,RUL_pred_mean,RUL_pred_median,RUL
0,1,150.125,157.5,112
1,2,143.625,138.0,98
2,3,70.875,53.0,69
3,4,82.000,71.5,82
4,5,85.625,77.0,91
5,6,93.375,92.0,93
6,7,83.875,85.0,91
7,8,61.625,55.5,95
8,9,163.250,163.5,111
9,10,65.125,64.5,96


In [20]:
rmse


{'mean': np.float64(25.987097158974876),
 'median': np.float64(27.737654911689994)}

In [21]:
sweep = sweep_k(result, compare_df)

NameError: name 'D_total' is not defined

In [ ]:
sweep

,k,rmse_mean,rmse_median
0,3,25.987097,27.737655
1,4,25.987097,27.737655
2,5,25.987097,27.737655
3,6,25.987097,27.737655
4,7,25.987097,27.737655
5,8,25.987097,27.737655
6,9,25.987097,27.737655
7,10,25.987097,27.737655
8,11,25.987097,27.737655
9,12,25.987097,27.737655


In [ ]:
def round_predictions_and_recount(df):
    """
    Hàm làm tròn các cột RUL dự đoán về số nguyên 
    và tính toán lại cột matches_count.
    """
    # 1. Làm tròn và ép kiểu về số nguyên (int)
    df['RUL_pred_mean'] = df['RUL_pred_mean'].round().astype(int)
    df['RUL_pred_median'] = df['RUL_pred_median'].round().astype(int)
    
    # 2. Đếm lại số lượng trùng khớp sau khi đã làm tròn
    df['matches_count_mean'] = (df['RUL_pred_mean'] == df['RUL']).astype(int)
    df['median']=(df['RUL_pred_median'] == df['RUL']).astype(int)
    
    return df

# Cách gọi hàm để áp dụng lên dataframe của bạn:
compare_df = round_predictions_and_recount(compare_df)

# Xem kết quả 5 dòng đầu
compare_df['matches_count_mean'].value_counts()


matches_count_mean
0    98
1     2
Name: count, dtype: int64

In [ ]:
compare_df['median'].value_counts()

median
0    99
1     1
Name: count, dtype: int64

In [ ]:
def calculate_error_range(df):
    # 1. Tính sai số cho giới hạn dưới và giới hạn trên, làm tròn và đưa về số nguyên
    lower_error = (df['RUL_pred_mean'] - df['RUL']).round().astype(int)
    upper_error = (df['RUL_pred_upper'] - df['RUL']).round().astype(int)
    
    # 2. Ghép chuỗi (string concatenation) để tạo định dạng [lower,upper]
    df['Range of Prediction Errors'] = '[' + lower_error.astype(str) + ',' + upper_error.astype(str) + ']'
    
    return df

# Áp dụng hàm
compare_df = calculate_error_range(compare_df)

# Xem kết quả
print(compare_df[['unit_number', 'Range of Prediction Errors']].head())

KeyError: 'RUL_pred_upper'